<a href="https://colab.research.google.com/github/luciaphmaestria/labo2025v/blob/main/src/ensembles/594_TareaHogar_05_1semilla_0.5Undersampling_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarea para el Hogar 05

Esta Tarea para el Hogar 05 se entrega el final de la cuarta clase
<br> se espera de usted que intente avanzar con los desafios propuestos y que los traiga terminados para la Clase 05 que será el miercoles 03 de septiembre

##  1. Overfitting the Public Leaderboard

Leer  https://medium.com/hmif-itb/overfitting-the-leaderboard-da25172ac62e
( 8 minutos )

## 2. Hiperparámetros del LightGBM

Los objetivos de esta tarea son:


*   Aumentar la rentabilidad de la campaña de marketing de retención proactiva de clientes.
*   Generar un mejor modelo optimizando sus hiperparámetros
*   Conceptual : investigar los mas relevantes hiperparámetros de LightGBM
*   Familiarizarse con la Bayesian Optimization, sus largos tiempos de corrida y opciones para reducirlos
*   Familiarizarse con el uso de máquinas virtuales de Google Colab
*   Ver un pipeline completo de optimización de hiperparámetros y puesta en producción

LightGBM cuenta con mas de 60 hiperparámetros, siendo posible utilizar 40 al mismo tiempo, aunque no razonable.
<br> La documentación oficial de los hiperparámetros de LightGBM es  https://lightgbm.readthedocs.io/en/latest/Parameters.html#core-parameters


Se lo alerta sobre que una Optimizacion Bayesiana lleva varias horas de corrida, y usted deberá correr VARIAS optimizaciones para descubrir cuales parámetros conviene optimizar.
<br> A pesar que la próxima clase es recien en viernes 01 de agosto, inicie la tarea con tiempo, aprenda a planificar estratégicamente sus corridas como un@ científ@  de datos.

Es necesario investigar cuales son los hiperparámetros de LightGBM que vale la pena optimizar en una Bayesian Optimization, ya que los realmente utiles son apenas un reducido subconjunto.
<br>Usted deberá investigar cuales son los hiperparámetros mas relevantes de LightGBM, su primer alternativa es preguntándole a su amigo con capacidades especiales ChatGPT o sus endogámicos familiares Claude, DeepSeek, Gemini, Grok, etc
<br> La segunda alternativa es la propia documentación de LightGBM  https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html


Adicionalmente podra buscar información como la que proveen esta diminuta muestra aleatoria de artículos ligeros:
* https://machinelearningmastery.com/light-gradient-boosted-machine-lightgbm-ensemble/
*  https://medium.com/@sarahzouinina/a-deep-dive-into-lightgbm-how-to-choose-and-tune-parameters-7c584945842e
*  https://www.kaggle.com/code/somang1418/tuning-hyperparameters-under-10-minutes-lgbm
*  https://towardsdatascience.com/beginners-guide-to-the-must-know-lightgbm-hyperparameters-a0005a812702/


<br>  La muestra anterior se brinda a modo de ejemplo, usted deberá buscar muuuuchas  fuentes adicionales de información
<br> Tenga presente que LightGBM es el estado del arte en modelado predictivo para datasets estructurado, que son el 90% del trabajo del 95% de los Data Scientists en Argentina.

El desafío de esta tarea es:
* Qué hiperparparámetros conviene optimizar?  Las recomendaciones de los artículos ligeros es siempre sensata?  Sus autores realmente hicieron experimentos o son siemplemente escritores de entretenimiento carente de base científica?
* Elegidos los hiperparámetros, cual es el  <desde, hasta> que se debe utilizar en la Bayesian Optimization ?
* Realmente vale la pena optimizar 10 o 16 hiperparámetros al mismo tiempo ?  No resulta contraproducente una búsqueda en un espacio de tal alta dimensionalidad ?

#### 2.1  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -s "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets



archivo_origen="https://storage.googleapis.com/open-courses/austral2025-af91/dataset_pequeno.csv"
archivo_destino="/content/datasets/dataset_pequeno.csv"
archivo_destino_bucket="/content/buckets/b1/datasets/dataset_pequeno.csv"

if ! test -f $archivo_destino_bucket; then
  wget  $archivo_origen  -O $archivo_destino_bucket
fi


if ! test -f $archivo_destino; then
  cp  $archivo_destino_bucket  $archivo_destino
fi


ln: failed to create symbolic link '/content/buckets/b1/labo1': File exists


### 2.2 Optimizacion Hiperparámetros

Esta parte se debe correr con el runtime en lenguaje R Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

### 2.2.1 Inicio

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Nov 04 03:15:29 AM 2025"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,660381,35.3,1454477,77.7,1454477,77.7
Vcells,1226627,9.4,8388608,64.0,1975128,15.1


### 2.2.2 Carga de Librerias

In [3]:
# cargo las librerias que necesito
require("data.table")
require("parallel")

if( !require("primes") ) install.packages("primes")
require("primes")

if( !require("utils") ) install.packages("utils")
require("utils")

if( !require("rlist") ) install.packages("rlist")
require("rlist")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm") ) install.packages("lightgbm")
require("lightgbm")

if( !require("DiceKriging") ) install.packages("DiceKriging")
require("DiceKriging")

if( !require("mlrMBO") ) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: data.table

Loading required package: parallel

Loading required package: primes

Loading required package: rlist

Loading required package: yaml

Loading required package: lightgbm

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers

Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘checkmate’


The following object is masked from ‘package:DiceKriging’:

    checkNames




### 2.2.3 Definicion de Parametros

aqui debe cargar SU semilla primigenia
<br>recuerde cambiar el numero de experimento en cada corrida nueva

In [4]:
PARAM <- list()
PARAM$experimento <- 5943
PARAM$semilla_primigenia <- 120539
PARAM$semilla_primigenia2 <- 230431


In [ ]:
PARAM$kaggle$competencia <- "labo-i-2025-virtual-analista-sr"
PARAM$kaggle$cortes <- seq(10000, 12000, by= 500)

In [5]:
# un undersampling de 0.1  toma solo el 10% de los CONTINUA
# undersampling de 1.0  implica tomar TODOS los datos

PARAM$trainingstrategy$undersampling <- 0.5                          # Lu. Le pongo 0.5 porque sigue siendo muy desbalanceada la clase

In [6]:
# Parametros LightGBM

PARAM$hyperparametertuning$xval_folds <- 5

# parametros fijos del LightGBM que se pisaran con la parte variable de la BO
PARAM$lgbm$param_fijos <-  list(
  boosting= "gbdt", # puede ir  dart  , ni pruebe random_forest
  objective= "binary",
  metric= "auc",
  first_metric_only= FALSE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  force_row_wise= TRUE, # para reducir warnings
  verbosity= -100,

  seed= PARAM$semilla_primigenia,

  max_depth= -1L, # -1 significa no limitar,  por ahora lo dejo fijo    # Dejo que BO ajuste por num_leaves
  min_gain_to_split= 0, # min_gain_to_split >= 0
  min_sum_hessian_in_leaf= 0.001, #  min_sum_hessian_in_leaf >= 0.0
  lambda_l1= 0.0, # lambda_l1 >= 0.0
  lambda_l2= 0.0, # lambda_l2 >= 0.0
  max_bin= 31L, # lo debo dejar fijo, no participa de la BO

  bagging_fraction= 1.0, # 0.0 < bagging_fraction <= 1.0                Lu. A optimizar
  bagging_freq=0,                                                       # Hiperparámetro agregado. A optimizar.
  pos_bagging_fraction= 1.0, # 0.0 < pos_bagging_fraction <= 1.0
  neg_bagging_fraction= 1.0, # 0.0 < neg_bagging_fraction <= 1.0
  is_unbalance= FALSE, #
  scale_pos_weight= 10, # scale_pos_weight > 0.0                       Lu. A optimizar. Primer valor tomo 10

  drop_rate= 0.1, # 0.0 < neg_bagging_fraction <= 1.0
  max_drop= 50, # <=0 means no limit
  skip_drop= 0.5, # 0.0 <= skip_drop <= 1.0

  extra_trees= FALSE,

  num_iterations= 1200,                                                #Lu. A optimizar
  learning_rate= 0.1,                                                  #Lu. A optimizar. Cambie valor original 0.02
  feature_fraction= 0.5,                                               #Lu. A optimizar
  num_leaves= 512,                                                     #Lu. A optimizar. Cambie valor original 750
  min_data_in_leaf= 300                                               #Lu. A optimizar. Cambie valor original 300
)


In [ ]:
print(PARAM)      #Lu

$experimento
[1] 5940

$semilla_primigenia
[1] 120539

$kaggle
$kaggle$competencia
[1] "labo-i-2025-virtual-analista-sr"

$kaggle$cortes
[1] 10000 10500 11000 11500 12000


$trainingstrategy
$trainingstrategy$undersampling
[1] 0.5


$hyperparametertuning
$hyperparametertuning$xval_folds
[1] 5


$lgbm
$lgbm$param_fijos
$lgbm$param_fijos$boosting
[1] "gbdt"

$lgbm$param_fijos$objective
[1] "binary"

$lgbm$param_fijos$metric
[1] "auc"

$lgbm$param_fijos$first_metric_only
[1] FALSE

$lgbm$param_fijos$boost_from_average
[1] TRUE

$lgbm$param_fijos$feature_pre_filter
[1] FALSE

$lgbm$param_fijos$force_row_wise
[1] TRUE

$lgbm$param_fijos$verbosity
[1] -100

$lgbm$param_fijos$seed
[1] 120539

$lgbm$param_fijos$max_depth
[1] -1

$lgbm$param_fijos$min_gain_to_split
[1] 0

$lgbm$param_fijos$min_sum_hessian_in_leaf
[1] 0.001

$lgbm$param_fijos$lambda_l1
[1] 0

$lgbm$param_fijos$lambda_l2
[1] 0

$lgbm$param_fijos$max_bin
[1] 31

$lgbm$param_fijos$bagging_fraction
[1] 1

$lgbm$param_fijos$pos_baggi

Aqui se definen los hiperparámetros de LightGBM que participan de la Bayesian Optimization
<br> si es un numero entero debe ir  makeIntegerParam
<br> si es un numero real (con decimales) debe ir  makeNumericParam
<br> es muy importante leer cuales son un lower y upper  permitidos y ademas razonables

In [7]:
# Aqui se cargan los bordes de los hiperparametros de la BO
PARAM$hypeparametertuning$hs <- makeParamSet(
  makeIntegerParam("num_iterations", lower= 120L, upper= 3000L),       # Lu. Rangos anteriores 8 - 2048
  makeNumericParam("learning_rate", lower= 0.005, upper= 0.1),          # Lu. Rangos anteriores 0.01 - 0.3
  makeNumericParam("feature_fraction", lower= 0.5, upper= 0.9),        # Lu. Rangos anteriores 0.1 - 1
  makeIntegerParam("min_data_in_leaf", lower= 150L, upper= 500L),       # Lu. Rangos anteriores 1 - 8000
  makeIntegerParam("num_leaves", lower= 8L, upper= 1024L),              # Lu. Rangos anteriores 8 - 2048
  makeNumericParam("bagging_fraction", lower= 0.6, upper= 1.0),        # Lu. Rangos anteriores 0.1 - 1
  makeIntegerParam("bagging_freq", lower= 0L, upper= 7L),             # Lu. Antes no estaba este parámetro
  makeIntegerParam("scale_pos_weight", lower= 1L, upper= 30L)         # Lu. Agregué para optimizar este hiperparámetro por clases altamente desbalanceadas.
)

A mayor cantidad de hiperparámetros, se debe aumentar las iteraciones de la Bayesian Optimization
<br> 30 es un valor muy tacaño, pero corre rápido
<br> deberia partir de 50, alcanzando los 100 si se dispone de tiempo

In [8]:
PARAM$hyperparametertuning$iteraciones <- 80    # iteraciones bayesianas. Es la cantidad de veces que prueba hiperparámetros (va iterando de forma inteligente, no prueba todas las combinaciones)

### 2.2.4  Preprocesamiento

In [9]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("HT", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [10]:
# lectura del dataset

dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
dataset

numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>
29183733,202107,1,0,0,46,99,2016.19,32146.69,1099.10,⋯,3,0.00,-27151.97,0.00,2995,10011.78,6,0,2498.49,CONTINUA
29184468,202107,1,0,0,42,284,1633.70,13781.99,602.61,⋯,10,0.00,-19268.21,0.00,5722,15138.18,9,0,3178.83,CONTINUA
29185245,202107,1,0,0,55,23,3759.73,24296.95,2829.38,⋯,3,0.00,-8211.00,0.00,758,58080.27,10,0,26240.01,CONTINUA
29186441,202107,1,0,0,62,296,2693.26,75390.56,2493.68,⋯,24,41585.05,-42723.62,0.00,8097,26541.92,8,0,1782.96,CONTINUA
29186475,202107,1,0,0,66,326,4567.63,56305.98,3690.64,⋯,3,0.00,-51675.11,0.00,2139,50304.17,16,0,3577.65,CONTINUA
29187730,202107,1,0,0,65,326,7943.33,131725.70,1166.58,⋯,24,93724.31,-75205.28,1.00,6516,81931.93,16,0,8480.79,CONTINUA
29187764,202107,1,0,0,56,91,4861.51,46598.35,1302.87,⋯,3,0.00,-207964.55,0.00,1719,22464.65,14,0,2486.76,CONTINUA
29187961,202107,1,0,0,62,326,6230.61,65485.56,4971.69,⋯,3,0.00,-222510.54,620.51,8516,98717.05,27,0,7706.61,CONTINUA
29189899,202107,1,0,0,60,379,3497.72,70109.88,2347.48,⋯,10,149469.00,-219762.79,0.00,9339,102991.71,43,0,8328.30,CONTINUA


In [11]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
dataset_train
str(dataset_train)

numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<int>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>
29183733,202107,1,0,0,46,99,2016.19,32146.69,1099.10,⋯,3,0.00,-27151.97,0.00,2995,10011.78,6,0,2498.49,CONTINUA
29184468,202107,1,0,0,42,284,1633.70,13781.99,602.61,⋯,10,0.00,-19268.21,0.00,5722,15138.18,9,0,3178.83,CONTINUA
29185245,202107,1,0,0,55,23,3759.73,24296.95,2829.38,⋯,3,0.00,-8211.00,0.00,758,58080.27,10,0,26240.01,CONTINUA
29186441,202107,1,0,0,62,296,2693.26,75390.56,2493.68,⋯,24,41585.05,-42723.62,0.00,8097,26541.92,8,0,1782.96,CONTINUA
29186475,202107,1,0,0,66,326,4567.63,56305.98,3690.64,⋯,3,0.00,-51675.11,0.00,2139,50304.17,16,0,3577.65,CONTINUA
29187730,202107,1,0,0,65,326,7943.33,131725.70,1166.58,⋯,24,93724.31,-75205.28,1.00,6516,81931.93,16,0,8480.79,CONTINUA
29187764,202107,1,0,0,56,91,4861.51,46598.35,1302.87,⋯,3,0.00,-207964.55,0.00,1719,22464.65,14,0,2486.76,CONTINUA
29187961,202107,1,0,0,62,326,6230.61,65485.56,4971.69,⋯,3,0.00,-222510.54,620.51,8516,98717.05,27,0,7706.61,CONTINUA
29189899,202107,1,0,0,60,379,3497.72,70109.88,2347.48,⋯,10,149469.00,-219762.79,0.00,9339,102991.71,43,0,8328.30,CONTINUA


Classes ‘data.table’ and 'data.frame':	164596 obs. of  155 variables:
 $ numero_de_cliente                   : int  29183733 29184468 29185245 29186441 29186475 29187730 29187764 29187961 29189899 29189993 ...
 $ foto_mes                            : int  202107 202107 202107 202107 202107 202107 202107 202107 202107 202107 ...
 $ active_quarter                      : int  1 1 1 1 1 1 1 1 1 1 ...
 $ cliente_vip                         : int  0 0 0 0 0 0 0 0 0 0 ...
 $ internet                            : int  0 0 0 0 0 0 0 0 0 0 ...
 $ cliente_edad                        : int  46 42 55 62 66 65 56 62 60 59 ...
 $ cliente_antiguedad                  : int  99 284 23 296 326 326 91 326 379 74 ...
 $ mrentabilidad                       : num  2016 1634 3760 2693 4568 ...
 $ mrentabilidad_annual                : num  32147 13782 24297 75391 56306 ...
 $ mcomisiones                         : num  1099 603 2829 2494 3691 ...
 $ mactivos_margen                     : num  -113 -241 746 -2909

In [12]:
# paso la clase a binaria que tome valores {0,1}  enteros     Agrega una nueva columna "clase01"
#  BAJA+1 y BAJA+2  son  1,   CONTINUA es 0

dataset_train[,
  clase01 := ifelse(clase_ternaria %in% c("BAJA+2","BAJA+1"), 1L, 0L)
]

In [13]:
# Contar por valor de clase01
dataset_train[, .N, by = clase01]

clase01,N
<int>,<int>
0,162211
1,2385


In [ ]:
dataset_train
str(dataset_train)

numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,⋯,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo,clase_ternaria,clase01
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<chr>,<int>
29183733,202107,1,0,0,46,99,2016.19,32146.69,1099.10,⋯,0.00,-27151.97,0.00,2995,10011.78,6,0,2498.49,CONTINUA,0
29184468,202107,1,0,0,42,284,1633.70,13781.99,602.61,⋯,0.00,-19268.21,0.00,5722,15138.18,9,0,3178.83,CONTINUA,0
29185245,202107,1,0,0,55,23,3759.73,24296.95,2829.38,⋯,0.00,-8211.00,0.00,758,58080.27,10,0,26240.01,CONTINUA,0
29186441,202107,1,0,0,62,296,2693.26,75390.56,2493.68,⋯,41585.05,-42723.62,0.00,8097,26541.92,8,0,1782.96,CONTINUA,0
29186475,202107,1,0,0,66,326,4567.63,56305.98,3690.64,⋯,0.00,-51675.11,0.00,2139,50304.17,16,0,3577.65,CONTINUA,0
29187730,202107,1,0,0,65,326,7943.33,131725.70,1166.58,⋯,93724.31,-75205.28,1.00,6516,81931.93,16,0,8480.79,CONTINUA,0
29187764,202107,1,0,0,56,91,4861.51,46598.35,1302.87,⋯,0.00,-207964.55,0.00,1719,22464.65,14,0,2486.76,CONTINUA,0
29187961,202107,1,0,0,62,326,6230.61,65485.56,4971.69,⋯,0.00,-222510.54,620.51,8516,98717.05,27,0,7706.61,CONTINUA,0
29189899,202107,1,0,0,60,379,3497.72,70109.88,2347.48,⋯,149469.00,-219762.79,0.00,9339,102991.71,43,0,8328.30,CONTINUA,0


Classes ‘data.table’ and 'data.frame':	164596 obs. of  156 variables:
 $ numero_de_cliente                   : int  29183733 29184468 29185245 29186441 29186475 29187730 29187764 29187961 29189899 29189993 ...
 $ foto_mes                            : int  202107 202107 202107 202107 202107 202107 202107 202107 202107 202107 ...
 $ active_quarter                      : int  1 1 1 1 1 1 1 1 1 1 ...
 $ cliente_vip                         : int  0 0 0 0 0 0 0 0 0 0 ...
 $ internet                            : int  0 0 0 0 0 0 0 0 0 0 ...
 $ cliente_edad                        : int  46 42 55 62 66 65 56 62 60 59 ...
 $ cliente_antiguedad                  : int  99 284 23 296 326 326 91 326 379 74 ...
 $ mrentabilidad                       : num  2016 1634 3760 2693 4568 ...
 $ mrentabilidad_annual                : num  32147 13782 24297 75391 56306 ...
 $ mcomisiones                         : num  1099 603 2829 2494 3691 ...
 $ mactivos_margen                     : num  -113 -241 746 -2909

In [14]:
# defino los datos que forma parte del training     # En este caso defini undesampling todo el dataset.
# aqui se hace el undersampling de los CONTINUA
# notar que para esto utilizo la SEGUNDA semilla

set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")   #Lu. "L'Ecuyer-CMRG" es un tipo de generador de números aleatorios que permite reproducibilidad.
                                                                  #Esto asegura que cada vez que corras el código, los mismos registros serán seleccionados en el undersampling.
dataset_train[, azar := runif(nrow(dataset_train))]          #Lu. Genera un vector de números aleatorios uniformes entre 0 y 1, de longitud igual al número de filas del dataset.
                                                                  #Dentro de data.table crea una nueva columna llamada azar con estos valores aleatorios.
                                                                  # Estacolumna servirá para seleccionar aleatoriamente qué filas de la clase mayoritaria se mantendrán en el entrenamiento.
dataset_train[, training := 0L]                              #Lu. Crea una columna training inicializada en 0 para marcar qué filas se usarán en el entrenamiento. 0L indica entero (tipo integer).

dataset_train[
  foto_mes %in% c(202107) &
    (azar <= PARAM$trainingstrategy$undersampling | clase_ternaria %in% c("BAJA+1", "BAJA+2")),  #Lu. selecciona aleatoriamente una fracción de las filas según el parámetro undersampling
                                                                                                      # asegura que todas las filas de las clases minoritarias se mantengan, sin importar el azar
  training := 1L   # Lu. Marca estas filas como parte del conjunto de entrenamiento.
]

In [15]:
# los campos que se van a utilizar

campos_buenos <- setdiff(
  colnames(dataset_train),
  c("clase_ternaria", "clase01", "azar", "training")
)

In [16]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[training == 1L, campos_buenos, with= FALSE]),
  label= dataset_train[training == 1L, clase01],
  free_raw_data= FALSE
)

nrow(dtrain)
ncol(dtrain)

[1] 83124

[1] 154

2.2.5 Configuracion Bayesian Optimization

In [17]:
# En el argumento x llegan los parmaetros de la bayesiana
#  devuelve la AUC en cross validation del modelo entrenado

EstimarGanancia_AUC_lightgbm <- function(x) {

  # Lu. Defino max_depth en función de num_leaves                                    # Lu.
  x$max_depth <- ceiling(log2(x$num_leaves + 1))                                     # +1 es un truco para asegurar que siempre haya suficiente profundidad, evitando que el árbol intente crecer más hojas de las que caben según max_depth.


  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelocv <- lgb.cv(
    data= dtrain,
    nfold= PARAM$hyperparametertuning$xval_folds,    # Lu. Quedó definido 5 folds. Valor confiable para la maroria de los datasets. No tiene sentido optimizarlo
    stratified= TRUE,                                # Lu. Importante porque hay clases desbalanceadas
    param= param_completo
  )

  # obtengo la ganancia
  AUC <- modelocv$best_score

  # hago espacio en la memoria
  rm(modelocv)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}

In [18]:
# Aqui comienza la configuracion de la Bayesian Optimization

# en este archivo quedan la evolucion binaria de la BO
kbayesiana <- "bayesiana.RDATA"

funcion_optimizar <- EstimarGanancia_AUC_lightgbm # la funcion que voy a maximizar

configureMlr(show.learner.output= FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo

obj.fun <- makeSingleObjectiveFunction(
  fn= funcion_optimizar, # la funcion que voy a maximizar
  minimize= FALSE, # estoy Maximizando la ganancia
  noisy= TRUE,
  par.set= PARAM$hypeparametertuning$hs, # definido al comienzo del programa.     Lu. Son los rangos de valores de los hiperparametros a optimizar
  has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
  save.on.disk.at.time= 600, # se graba cada 600 segundos
  save.file.path= kbayesiana
) # se graba cada 600 segundos

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
  ctrl,
  iters= PARAM$hyperparametertuning$iteraciones
) # cantidad de iteraciones

# defino el método estandar para la creacion de los puntos iniciales,
# los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit= makeMBOInfillCritEI())

# establezco la funcion que busca el maximo
surr.km <- makeLearner(
  "regr.km",
  predict.type= "se",
  covtype= "matern3_2",
  control= list(trace= TRUE)
)


2.2.6 Corrida Bayesian Optimization

In [19]:
# inicio la optimizacion bayesiana, retomando si ya existe
# es la celda mas lenta de todo el notebook

if (!file.exists(kbayesiana)) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue(kbayesiana) # retomo en caso que ya exista
}

Tue Nov 04 03:17:57 AM 2025 AUC 0.926243592870419

[mbo] 11: num_iterations=463; learning_rate=0.00546; feature_fraction=0.542; min_data_in_leaf=443; num_leaves=1022; bagging_fraction=0.868; bagging_freq=3; scale_pos_weight=16 : y = 0.926 : 87.8 secs : infill_ei

Saved the current state after iteration 12 in the file bayesiana.RDATA.

Tue Nov 04 03:24:15 AM 2025 AUC 0.930680371536239

[mbo] 12: num_iterations=2221; learning_rate=0.00756; feature_fraction=0.834; min_data_in_leaf=435; num_leaves=1023; bagging_fraction=0.943; bagging_freq=2; scale_pos_weight=15 : y = 0.931 : 378.0 secs : infill_ei

Tue Nov 04 03:28:47 AM 2025 AUC 0.929215980403803

[mbo] 13: num_iterations=1991; learning_rate=0.0109; feature_fraction=0.696; min_data_in_leaf=430; num_leaves=969; bagging_fraction=0.841; bagging_freq=5; scale_pos_weight=13 : y = 0.929 : 271.1 secs : infill_ei

Saved the current state after iteration 14 in the file bayesiana.RDATA.

Tue Nov 04 03:35:54 AM 2025 AUC 0.929825748002504

[mbo] 14:

In [ ]:

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)
colnames( tb_bayesiana)

[1] "num_iterations"   "learning_rate"    "feature_fraction" "min_data_in_leaf"
 [5] "num_leaves"       "bagging_fraction" "bagging_freq"     "scale_pos_weight"
 [9] "y"                "dob"              "eol"              "error.message"   
[13] "exec.time"        "ei"               "error.model"      "train.time"      
[17] "prop.type"        "propose.time"     "se"               "mean"

In [ ]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

tb_bayesiana[, iter := .I]

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file= "BO_log.txt",
  sep= "\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  setdiff(colnames(tb_bayesiana),
    c("y","dob","eol","error.message","exec.time","ei","error.model",
      "train.time","prop.type","propose.time","se","mean","iter")),
  with= FALSE
]


PARAM$out$lgbm$y <- tb_bayesiana[1, y]


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
print(PARAM$out$lgbm$mejores_hiperparametros)
print(PARAM$out$lgbm$y)

   num_iterations learning_rate feature_fraction min_data_in_leaf num_leaves
            <int>         <num>            <num>            <int>      <int>
1:            585    0.01156296        0.5956823              268        432
   bagging_fraction bagging_freq scale_pos_weight
              <num>        <int>            <int>
1:        0.8326237            1               10
[1] 0.9318529


## 2.3  Produccion

### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

In [ ]:
setwd("/content/buckets/b1/exp")
experimento <- paste0("exp", PARAM$experimento)
dir.create(experimento, showWarnings= FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

#### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [ ]:
# clase01
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

In [ ]:
dataset_train <- dataset[foto_mes %in% c(202107)]

In [ ]:
# dejo los datos en el formato que necesita LightGBM

dtrain <- lgb.Dataset(
  data= data.matrix(dataset_train[, campos_buenos, with= FALSE]),
  label= dataset_train[, clase01]
)

#### Final Training Hyperparameters

In [ ]:
param_final <- modifyList(PARAM$lgbm$param_fijos,
  PARAM$out$lgbm$mejores_hiperparametros)

param_final

$boosting
[1] "gbdt"

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] FALSE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$force_row_wise
[1] TRUE

$verbosity
[1] -100

$seed
[1] 120539

$max_depth
[1] -1

$min_gain_to_split
[1] 0

$min_sum_hessian_in_leaf
[1] 0.001

$lambda_l1
[1] 0

$lambda_l2
[1] 0

$max_bin
[1] 31

$bagging_fraction
[1] 0.8326237

$bagging_freq
[1] 1

$pos_bagging_fraction
[1] 1

$neg_bagging_fraction
[1] 1

$is_unbalance
[1] FALSE

$scale_pos_weight
[1] 10

$drop_rate
[1] 0.1

$max_drop
[1] 50

$skip_drop
[1] 0.5

$extra_trees
[1] FALSE

$num_iterations
[1] 585

$learning_rate
[1] 0.01156296

$feature_fraction
[1] 0.5956823

$num_leaves
[1] 432

$min_data_in_leaf
[1] 268

#### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [ ]:
# este punto es muy SUTIL  y será revisado en la Clase 05

param_normalizado <- copy(param_final)
param_normalizado$min_data_in_leaf <-  round(param_final$min_data_in_leaf / PARAM$trainingstrategy$undersampling)

In [ ]:
  # entreno LightGBM

  modelo_final <- lgb.train(
    data= dtrain,
    param= param_normalizado
  )

In [ ]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(modelo_final))
archivo_importancia <- "impo.txt"

fwrite(tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

In [ ]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(modelo_final, "modelo.txt" )

### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
# aplico el modelo a los datos sin clase
dfuture <- dataset[foto_mes == 202109]

# aplico el modelo a los datos nuevos
prediccion <- predict(
  modelo_final,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

#### Tabla Prediccion

In [ ]:
# tabla de prediccion

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion ]

# grabo las probabilidad del modelo
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

Kaggle Competition Submit

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
  Sys.sleep(45)
}

Successfully submitted to LaboI 2025 virtual analista sr 
Successfully submitted to LaboI 2025 virtual analista sr 
Successfully submitted to LaboI 2025 virtual analista sr 
Successfully submitted to LaboI 2025 virtual analista sr 
Successfully submitted to LaboI 2025 virtual analista sr 


In [ ]:
write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Nov 03 05:57:50 PM 2025"

Finalmente usted deberá cargar el resultado de su corrida en la Google Sheet Colaborativa,  hoja **TareaHogar-05**
<br> Siéntase libre de agregar las columnas que hagan falta a la planilla

Seguramente usted realice varias corridas de este script con distintos conjuntos de hiperparámetros, siempre cambiandole el nombre al script  y también cambiando el nombre del experimento,  deberá TODAS esas corridas en distintas lineas de la  Google Sheet Colaborativa, hoja **TareaHogar-05**

Siéntase libre de agregar columnas a la hoja **TareaHogar-05**  en caso de ser necesario.